# WoundMap AI: Phase 5 — Evaluation & Error Analysis

**Project**: WoundMap AI (Biophysics & Biomedical Computer-Vision Research Prototype)  
**Governing Blueprint**: `WoundMapAI_Antigravity_Autonomous_Execution_Blueprint.pdf`  
**Evaluation Set**: Strictly untouched test partition (`data/manifests/test.csv`, $N=2$, 131,072 pixels)  
**Checkpoint**: `outputs/checkpoints/baseline/best_baseline_unet_rgb.pt`

---

### Evaluation Safeguards
1. **Untouched Test Manifest**: No hyperparameters or weights are tuned on these test results.
2. **Dual Macro Averaging**: Both present-class and all-class macro averages are reported explicitly.
3. **Absent-Class Disclosure**: Absent classes (Bone, Tendon) have support = 0 and are documented transparently.

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
import numpy as np
import pandas as pd
from PIL import Image

import src.woundmap.compat
from src.woundmap.models import build_model
from src.woundmap.data import build_dataloader
from src.woundmap.masks import mask_to_rgb, WOUNDTISSUE_PALETTE
from src.woundmap.metrics import compute_classification_metrics, benchmark_inference_latency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

## 1. Ingest Checkpoint and Test Dataloader

In [ ]:
ckpt_path = os.path.join("..", "outputs", "checkpoints", "baseline", "best_baseline_unet_rgb.pt")
model = build_model("unet", in_channels=3, num_classes=7, base_features=16).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_loader = build_dataloader(
    manifest_csv=os.path.join("..", "data", "manifests", "test.csv"),
    batch_size=1,
    shuffle=False,
    target_size=(256, 256),
    color_space="RGB",
    illumination_norm="none",
    augment=False,
    seed=42
)

print(f"Loaded test set with {len(test_loader)} images.")

## 2. Quantitative Evaluation: Per-Class & Macro Metrics

In [ ]:
metrics_csv = os.path.join("..", "reports", "test_metrics_per_class.csv")
df_metrics = pd.read_csv(metrics_csv)
display(df_metrics)

summary_json = os.path.join("..", "reports", "test_metrics_summary.json")
with open(summary_json) as f:
    import json
    summary = json.load(f)

print(f"Macro Dice (Present Classes): {summary['macro_dice_present']:.4f}")
print(f"Macro Dice (All 7 Classes):   {summary['macro_dice_all']:.4f}")
print(f"Mean Latency:                {summary['latency_benchmark']['mean_ms']:.2f} ms ({summary['latency_benchmark']['fps']:.1f} FPS)")

## 3. Confusion Matrix and Boundary Error Analysis
- **Heatmap**: Displays pixel transitions between ground-truth and predicted tissue categories.
- **Primary Failure Mode**: Confusion between low-reflectance necrotic tissue and background shadows in standard sRGB.
- **Hypothesis Motivation for Phase 6**: Applying CIELAB/HSV color decoupling with CLAHE on luminance is specifically designed to eliminate this dark-tissue/shadow ambiguity.